# Sentiment Analysis using NLP Pipeline and Machine Learning


In [10]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aravi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aravi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Data Understanding

In this section we load the dataset and explore:

• Number of samples  
• Class distribution  
• Sample reviews

In [11]:
df=pd.read_csv("IMDB Dataset.csv")

df=df.sample(10000,random_state=42)

df.head()

,review,sentiment
33553,I really liked this Summerslam due to the look...,positive
9427,Not many television shows appeal to quite as m...,positive
199,The film quickly gets to a major chase scene w...,negative
12447,Jane Austen would definitely approve of this o...,positive
39489,Expectations were somewhat high for me when I ...,negative


In [12]:
df.shape

(10000, 2)

In [13]:
df['sentiment'].value_counts()

sentiment
positive    5039
negative    4961
Name: count, dtype: int64

In [33]:
df.isnull().sum()

review          0
sentiment       0
clean_review    0
dtype: int64

In [14]:
df.sample(5)

,review,sentiment
32279,This Is one of my favourite westerns. What a c...,positive
46683,Although in some aspects Seven Pounds is solid...,negative
11980,"""The Student Nurses"" is an excellent film that...",positive
34407,"This movie tries hard, but completely lacks th...",negative
23783,I spotted this film in a branch of the Duane R...,positive


# NLP Preprocessing

Steps performed:

• Lowercasing  
• Removing punctuation  
• Removing stopwords  
• Tokenization  
• Stemming  
• Removing URLs  

Reusable preprocessing function is created.

In [15]:
stop_words=set(stopwords.words('english'))

ps=PorterStemmer()

def clean_text(text):

    text=str(text).lower()

    text=re.sub(r"http\S+","",text)

    text=re.sub(r"[^a-zA-Z ]","",text)

    words=text.split()

    words=[word for word in words if word not in stop_words]

    words=[ps.stem(word) for word in words]

    return " ".join(words)

In [16]:
df['clean_review']=df['review'].apply(clean_text)
df.head()

,review,sentiment,clean_review
33553,I really liked this Summerslam due to the look...,positive,realli like summerslam due look arena curtain ...
9427,Not many television shows appeal to quite as m...,positive,mani televis show appeal quit mani differ kind...
199,The film quickly gets to a major chase scene w...,negative,film quickli get major chase scene ever increa...
12447,Jane Austen would definitely approve of this o...,positive,jane austen would definit approv onebr br gwyn...
39489,Expectations were somewhat high for me when I ...,negative,expect somewhat high went see movi thought ste...


# Feature Engineering

We convert text into numerical features using:

• Bag of Words  
• TF-IDF

In [17]:
bow=CountVectorizer(max_features=5000)

X_bow=bow.fit_transform(df['clean_review'])

In [19]:
tfidf=TfidfVectorizer(max_features=5000)

X_tfidf=tfidf.fit_transform(df['clean_review'])

In [20]:
y=df['sentiment']

y=y.map({'positive':1,'negative':0})

In [23]:
#train test split
X_train_bow,X_test_bow,y_train,y_test=train_test_split(
X_bow,y,test_size=0.2,random_state=42)

X_train_tfidf,X_test_tfidf,y_train,y_test=train_test_split(
X_tfidf,y,test_size=0.2,random_state=42)

# Model Building

Models used:

• Logistic Regression  
• Naive Bayes  
• Decision Tree

In [26]:
#Logistic Regression
lr=LogisticRegression()

lr.fit(X_train_tfidf,y_train)

pred_lr=lr.predict(X_test_tfidf)

In [27]:
#Naive Bayes
nb=MultinomialNB()

nb.fit(X_train_bow,y_train)

pred_nb=nb.predict(X_test_bow)

In [28]:
#Decision Tree
dt=DecisionTreeClassifier()

dt.fit(X_train_tfidf,y_train)

pred_dt=dt.predict(X_test_tfidf)

# Model Evaluation

Models are evaluated using:

• Accuracy  
• Precision  
• Recall  
• F1 Score

In [29]:
#Logistic Regression evaluation
accuracy_score(y_test,pred_lr)

print(classification_report(y_test,pred_lr))

              precision    recall  f1-score   support

           0       0.88      0.84      0.86       999
           1       0.85      0.89      0.87      1001

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000



In [30]:
#Naive Bayes evaluation
accuracy_score(y_test,pred_nb)

print(classification_report(y_test,pred_nb))

              precision    recall  f1-score   support

           0       0.83      0.86      0.84       999
           1       0.85      0.82      0.84      1001

    accuracy                           0.84      2000
   macro avg       0.84      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000



In [31]:
#Decision Tree evaluation
accuracy_score(y_test,pred_dt)

print(classification_report(y_test,pred_dt))

              precision    recall  f1-score   support

           0       0.70      0.71      0.70       999
           1       0.70      0.70      0.70      1001

    accuracy                           0.70      2000
   macro avg       0.70      0.70      0.70      2000
weighted avg       0.70      0.70      0.70      2000



In [32]:
#Accuracy comparison table
results=pd.DataFrame({

'Model':['Logistic Regression','Naive Bayes','Decision Tree'],

'Accuracy':[

accuracy_score(y_test,pred_lr),

accuracy_score(y_test,pred_nb),

accuracy_score(y_test,pred_dt)

]

})

results

,Model,Accuracy
0,Logistic Regression,0.864
1,Naive Bayes,0.841
2,Decision Tree,0.703
